# 第 13 章实验：去噪自编码器与表示相似度
合成条纹图、训练极小去噪 AE，再比较同一样本不同噪声视图的编码相似度。Notebook 不给出 VAE 或 NT-Xent 作业答案。

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(13)
samples = 64
size = 16
noise_std = 0.30
train_steps = 70

In [ ]:
clean = torch.zeros(samples, 1, size, size)
for i in range(samples):
    offset = 3 + (i % 6)
    if i % 2 == 0:
        clean[i, 0, :, offset:offset+3] = 1
    else:
        clean[i, 0, offset:offset+3, :] = 1
clean = (clean + 0.03 * torch.randn_like(clean)).clamp(0, 1)
noisy = (clean + noise_std * torch.randn_like(clean)).clamp(0, 1)
print('clean/noisy:', tuple(clean.shape), clean.min().item(), clean.max().item())

In [ ]:
class DenoisingAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(4, 8, 3, padding=1), nn.ReLU())
        self.decoder = nn.Sequential(nn.Conv2d(8, 4, 3, padding=1), nn.ReLU(), nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False), nn.Conv2d(4, 1, 3, padding=1), nn.Sigmoid())
    def forward(self, x):
        return self.decoder(self.encoder(x))

model = DenoisingAE()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
history = []
for _ in range(train_steps):
    optimizer.zero_grad(); reconstruction = model(noisy); loss = F.mse_loss(reconstruction, clean)
    loss.backward(); optimizer.step(); history.append(float(loss.detach()))
with torch.no_grad(): reconstruction = model(noisy)
print(f'final reconstruction MSE={history[-1]:.5f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
axes[0].imshow(clean[0,0], cmap='gray'); axes[0].set_title('clean')
axes[1].imshow(noisy[0,0], cmap='gray'); axes[1].set_title('noisy')
axes[2].imshow(reconstruction[0,0], cmap='gray'); axes[2].set_title('reconstruction')
axes[3].plot(history); axes[3].set_title('MSE loss')
for ax in axes[:3]: ax.axis('off')
plt.tight_layout(); plt.show()

## 自监督表示观察
把同一干净图加入两份独立噪声，比较编码向量余弦相似度；再与另一类别样本比较。这里只观察，不实现对比损失。

In [ ]:
view1 = (clean[:8] + noise_std * torch.randn_like(clean[:8])).clamp(0, 1)
view2 = (clean[:8] + noise_std * torch.randn_like(clean[:8])).clamp(0, 1)
with torch.no_grad():
    z1 = model.encoder(view1).flatten(1)
    z2 = model.encoder(view2).flatten(1)
    same = F.cosine_similarity(z1, z2).mean()
    shifted = F.cosine_similarity(z1, z2.roll(1, dims=0)).mean()
print(f'same-image similarity={float(same):.3f}')
print(f'shifted-pair similarity={float(shifted):.3f}')
print('注意：重建好不保证下游表示一定好，仍需线性探测。')

### 破坏实验建议
把噪声增大到 0.8，或把瓶颈通道扩得很宽。前者可能破坏语义，后者可能学成近似复制；分别记录重建与表示相似度。